In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/workspace/tiny-llm-from-scratch")

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/workspace/tiny-llm-from-scratch


In [5]:
from training.dataset import TinyDataset
from training.sequence import SequenceBuilder

from tokenizer.bpe_tokenizer import BPETokenizer

In [6]:
dataset = TinyDataset.load(
    PROJECT_ROOT /
    "datasets/processed/split/train.json"
)

print("Samples:", len(dataset))

Samples: 4783


In [7]:
tokenizer_path = (
    PROJECT_ROOT /
    "tokenizer/bpe/tokenizer.json"
)

print("Tokenizer File:")
print(tokenizer_path)

print()

print("Exists:", tokenizer_path.exists())

tokenizer = BPETokenizer.load(
    tokenizer_path
)

print()

print("✓ BPETokenizer Loaded Successfully")

Tokenizer File:
/workspace/tiny-llm-from-scratch/tokenizer/bpe/tokenizer.json

Exists: True

✓ BPETokenizer Loaded Successfully


In [8]:
builder = SequenceBuilder(
    tokenizer=tokenizer,
    template="default",
)

print("✓ SequenceBuilder Ready")

✓ SequenceBuilder Ready


In [9]:
sample = dataset[0]

print(sample)

Sample(instruction='Refactor and harden an existing implementation of distributed actor system in Python using asyncio and Ray for maximum security and performance. Show before/after metrics and formal verification steps.', response='Drawing from the autonomous, frontier-level reasoning characteristic of Claude Mythos (distilled for accessibility and precision), I approach this with multi-layered analysis, rigorous decomposition, security-first mindset, performance considerations, and ethical guardrails.\n\n**Mythos-Grade Implementation Strategy**\n\n**Problem Decomposition**\n- Performance target: >10M operations/sec on commodity hardware\n- Constraints: constant memory footprint\n- Non-functional: Observability, graceful degradation, formal memory safety\n\n**Production-Ready Code (C++20)**\n\n``` c++20\n// Zero-copy, SIMD-accelerated protocol parser\n#[inline(always)]\npub fn parse_frame(data: &[u8]) -> Result<Frame, ParseError> {\n // Mythos-level branchless parsing with prefetch h

In [10]:
encoded = builder.encode_sample(sample)

print("Token Count:", encoded.length)

Token Count: 777


In [11]:
builder.preview(encoded)

ENCODED SAMPLE

User: Refactor and harden an existing implementation of distributed actor system in Python using asyncio and Ray for maximum security and performance. Show before/after metrics and formal verification steps.

Assistant: Drawing from the autonomous, frontier-level reasoning characteristic of Claude Mythos (distilled for accessibility and precision), I approach this with multi-layered analysis, rigorous decomposition, security-first mindset, performance considerations, and ethical guardrails.

**Mythos-Grade Implementation Strategy**

**Problem Decomposition**
- Performance target: >10M operations/sec on commodity hardware
- Constraints: constant memory footprint
- Non-functional: Observability, graceful degradation, formal memory safety

**Production-Ready Code (C++20)**

``` c++20
// Zero-copy, SIMD-accelerated protocol parser
#[inline(always)]
pub fn parse_frame(data: &[u8]) -> Result<Frame, ParseError> {
 // Mythos-level branchless parsing with prefetch hints
 ...
}
`

In [12]:
encoded_samples = []

for sample in dataset.head(20):

    encoded_samples.append(
        builder.encode_sample(sample)
    )

print("Encoded:", len(encoded_samples))

Encoded: 20


In [13]:
stats = builder.statistics(
    encoded_samples
)

stats

{'samples': 20,
 'tokens': 17693,
 'min_length': 754,
 'max_length': 1125,
 'avg_length': 884.65}

In [14]:
longest = max(
    encoded_samples,
    key=lambda x: x.length,
)

print(longest.length)

1125


In [15]:
print("=" * 60)

print("Sequence Builder Test")

print("=" * 60)

print("Dataset Samples :", len(dataset))

print("Encoded Samples :", len(encoded_samples))

print("Average Length  :", round(stats["avg_length"], 2))

print("Maximum Length  :", stats["max_length"])

print()

print("★★★★★ Phase 11.05.01 PASSED")

print("=" * 60)

Sequence Builder Test
Dataset Samples : 4783
Encoded Samples : 20
Average Length  : 884.65
Maximum Length  : 1125

★★★★★ Phase 11.05.01 PASSED


In [16]:
pairs = builder.build_training_pairs(
    encoded.token_ids,
    block_size=64,
    stride=1,
)

print("=" * 60)
print("Training Pairs")
print("=" * 60)

print("Total Pairs :", len(pairs))

Training Pairs
Total Pairs : 713


In [17]:
builder.preview_pairs(
    pairs,
    count=3,
)

GPT TRAINING PAIRS

Pair 1
----------------------------------------------------------------------
Input (x)
[13, 8, 8, 5, 6, 14, 12, 5, 7, 9, 5, 20, 5, 8, 13, 14, 11, 10, 8, 11, 8, 14, 5, 14, 12, 12, 7, 13, 14, 15, 14, 8, 7, 5, 6, 14, 12, 13, 16, 13, 14, 8, 11, 23, 15, 13, 5, 13, 16, 6, 12, 5, 7, 5, 16, 12, 11, 5, 11, 15, 11, 13, 8, 6]

Target (y)
[8, 8, 5, 6, 14, 12, 5, 7, 9, 5, 20, 5, 8, 13, 14, 11, 10, 8, 11, 8, 14, 5, 14, 12, 12, 7, 13, 14, 15, 14, 8, 7, 5, 6, 14, 12, 13, 16, 13, 14, 8, 11, 23, 15, 13, 5, 13, 16, 6, 12, 5, 7, 5, 16, 12, 11, 5, 11, 15, 11, 13, 8, 6, 15]

Decoded Input
s e e a c t o a d h a de a e s t m l e m e t a t o o d s t u t e d a c t o s y s t e m ytho u s a s y c o a d a y o m a m u m s e c

Decoded Target
e e a c t o a d h a de a e s t m l e m e t a t o o d s t u t e d a c t o s y s t e m ytho u s a s y c o a d a y o m a m u m s e c u

Pair 2
----------------------------------------------------------------------
Input (x)
[8, 8, 5, 6, 14, 12, 5, 7, 9, 5, 20,

In [18]:
training_dataset = builder.build_dataset(
    encoded_samples,
    block_size=64,
    stride=1,
)

print("=" * 60)
print("Training Dataset")
print("=" * 60)

print("Total Training Examples :", len(training_dataset))

Training Dataset
Total Training Examples : 16413


In [19]:
first_pair = training_dataset[0]

print("=" * 60)
print("First Training Pair")
print("=" * 60)

print("Input Length :", len(first_pair.x))
print("Target Length:", len(first_pair.y))

print()

print("Input")

print(first_pair.x)

print()

print("Target")

print(first_pair.y)

First Training Pair
Input Length : 64
Target Length: 64

Input
[13, 8, 8, 5, 6, 14, 12, 5, 7, 9, 5, 20, 5, 8, 13, 14, 11, 10, 8, 11, 8, 14, 5, 14, 12, 12, 7, 13, 14, 15, 14, 8, 7, 5, 6, 14, 12, 13, 16, 13, 14, 8, 11, 23, 15, 13, 5, 13, 16, 6, 12, 5, 7, 5, 16, 12, 11, 5, 11, 15, 11, 13, 8, 6]

Target
[8, 8, 5, 6, 14, 12, 5, 7, 9, 5, 20, 5, 8, 13, 14, 11, 10, 8, 11, 8, 14, 5, 14, 12, 12, 7, 13, 14, 15, 14, 8, 7, 5, 6, 14, 12, 13, 16, 13, 14, 8, 11, 23, 15, 13, 5, 13, 16, 6, 12, 5, 7, 5, 16, 12, 11, 5, 11, 15, 11, 13, 8, 6, 15]


In [20]:
assert len(dataset) > 0

assert len(encoded_samples) > 0

assert len(training_dataset) > 0

assert len(first_pair.x) == 64

assert len(first_pair.y) == 64

assert len(first_pair.x) == len(first_pair.y)

print()

print("✓ Dataset Loaded")

print("✓ Samples Encoded")

print("✓ GPT Pairs Created")

print("✓ Block Size Correct")

print("✓ Assertions Passed")


✓ Dataset Loaded
✓ Samples Encoded
✓ GPT Pairs Created
✓ Block Size Correct
✓ Assertions Passed


In [21]:
lengths = [
    len(pair.x)
    for pair in training_dataset
]

print("=" * 60)
print("Dataset Statistics")
print("=" * 60)

print("Minimum Block :", min(lengths))
print("Maximum Block :", max(lengths))
print("Average Block :", sum(lengths) / len(lengths))

Dataset Statistics
Minimum Block : 64
Maximum Block : 64
Average Block : 64.0


In [22]:
import time

start = time.time()

_ = builder.build_dataset(
    encoded_samples,
    block_size=64,
    stride=1,
)

elapsed = time.time() - start

print("=" * 60)
print("Performance")
print("=" * 60)

print(f"Dataset Build Time : {elapsed:.3f} sec")

Performance
Dataset Build Time : 0.137 sec


In [23]:
print("=" * 60)

print("TinyLLM Sequence Builder Validation")

print("=" * 60)

print(f"Dataset Samples     : {len(dataset)}")

print(f"Encoded Samples     : {len(encoded_samples)}")

print(f"Training Examples   : {len(training_dataset)}")

print(f"Block Size          : {len(first_pair.x)}")

print(f"Tokenizer           : BPE")

print()

print("★★★★★ Phase 11.05 PASSED")

print("=" * 60)

TinyLLM Sequence Builder Validation
Dataset Samples     : 4783
Encoded Samples     : 20
Training Examples   : 16413
Block Size          : 64
Tokenizer           : BPE

★★★★★ Phase 11.05 PASSED
